creation du database financecore_db

In [ ]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

load_dotenv(dotenv_path=".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DEFAULT_DB = os.getenv("DEFAULT_DB")
NEW_DB = os.getenv("NEW_DB")
DB_NAME = os.getenv("DB_NAME")


default_db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres"
engine = create_engine(default_db_url)



with engine.connect() as conn:
    conn.execution_options(isolation_level="AUTOCOMMIT")
    
    result = conn.execute(text(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'"))
    
    if not result.scalar():
        conn.execute(text(f"CREATE DATABASE {DB_NAME}"))
        print(f"Database '{DB_NAME}' created successfully.")
    else:
        print(f"Database '{DB_NAME}' already exists.")


Database 'financecore_db' created successfully.


In [ ]:
import os
import urllib.parse
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(dotenv_path="financecore_db.env")

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

encoded_pass = urllib.parse.quote_plus(DB_PASS)
DATABASE_URL = f"postgresql+psycopg://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

def run_health_check():
    print("🔍 System integrity check ...")
    try:
        with engine.connect() as conn:
            res = conn.execute(text("SELECT current_user, current_database(), version();"))
            user, db, ver = res.fetchone()
            print(f"✅ Connection: Stable")
            print(f"👤 User     : {user}")
            print(f"🗄️ Database : {db}")

            conn.execute(text("CREATE TEMP TABLE connection_test (id serial PRIMARY KEY, val text);"))
            conn.execute(text("INSERT INTO connection_test (val) VALUES ('test');"))
            conn.execute(text("DROP TABLE connection_test;"))
            print("✅ Permissions: All necessary permissions are in place.")
            print("\n🚀 System is ready to proceed to the table creation stage!")

    except SQLAlchemyError as e:
        print("\n❌ System integrity check failed")
        print(f"⚠️ Technical reason: {e}")

if __name__ == "__main__":
    run_health_check()